# TensorFlow 실습 001 — 와인 분류 신경망

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/tensorflow_001.ipynb)

**머신러닝 with 파이썬 12.3.7절 — 분류 신경망 실습**

신경망을 이용해 와인 데이터를 3가지 종류로 분류합니다.

### 이 노트북에서 배울 것
1. 데이터 로드 및 전처리 (원-핫 인코딩, train/test 분할)
2. `Sequential` 모델로 신경망 구성
3. `Dense`, `BatchNormalization`, `Activation` 레이어의 역할
4. 모델 컴파일 (`loss`, `optimizer`, `metrics`)
5. 학습 및 평가
6. 정확도/손실 그래프 시각화

---
## Cell 1. 라이브러리 임포트

딥러닝에 필요한 라이브러리들을 불러옵니다. 각 라이브러리가 어떤 역할인지 주석으로 설명합니다.

In [ ]:
# ============================================================
# 데이터 관련 라이브러리
# ============================================================

# sklearn.datasets: 사이킷런에 내장된 샘플 데이터셋 모음
# - load_wine(), load_iris(), load_digits() 등 학습용 데이터 제공
from sklearn import datasets

# to_categorical: 정수 레이블을 원-핫 인코딩(One-Hot Encoding)으로 변환
# - 예: 2 → [0, 0, 1]  (3클래스 기준)
# - 딥러닝 분류 문제에서 타깃(y)을 이 형태로 변환해야 함
from tensorflow.keras.utils import to_categorical

# train_test_split: 데이터를 학습용(train)과 평가용(test)으로 분할
# - 보통 75:25 또는 80:20 비율로 나눔
# - 모델이 "처음 보는 데이터"에 대해 얼마나 잘 동작하는지 확인하기 위함
from sklearn.model_selection import train_test_split

# numpy: 수치 계산 라이브러리 (배열, 행렬 연산)
import numpy as np

# tensorflow: 구글이 만든 딥러닝 프레임워크
# - 텐서(다차원 배열) 연산, 자동 미분, GPU 가속 지원
import tensorflow as tf

# ============================================================
# 신경망 구성 관련 라이브러리
# ============================================================

# Sequential: 레이어를 순서대로 쌓는 가장 기본적인 모델 구조
# - model.add()로 레이어를 하나씩 추가하는 방식
# - 입력 → 레이어1 → 레이어2 → ... → 출력 (일직선 구조)
from tensorflow.keras.models import Sequential

# Input: 모델의 입력 형태(shape)를 명시적으로 정의하는 레이어
# - Keras 최신 버전에서는 Dense(input_dim=...) 대신 Input()을 권장
# - 경고(UserWarning) 없이 깔끔하게 입력 차원을 지정
from tensorflow.keras.layers import Input

# Dense: 완전연결층(Fully Connected Layer)
# - 이전 레이어의 모든 뉴런과 현재 레이어의 모든 뉴런이 연결
# - 수식: output = activation(W·input + b)
#   - W: 가중치 행렬 (학습으로 업데이트)
#   - b: 편향 벡터 (학습으로 업데이트)
from tensorflow.keras.layers import Dense

# BatchNormalization: 배치 정규화
# - 각 미니배치의 입력을 정규화(평균=0, 분산=1)하여 학습을 안정화
# - 효과: 학습 속도 향상, 기울기 소실/폭발 완화, 더 높은 학습률 사용 가능
# - 수식: x_norm = (x - μ_batch) / √(σ²_batch + ε)
#         output = γ · x_norm + β  (γ, β는 학습 가능한 파라미터)
from tensorflow.keras.layers import BatchNormalization

# Activation: 활성화 함수 레이어
# - 신경망에 비선형성(nonlinearity)을 부여
# - 활성화 함수가 없으면 아무리 레이어를 쌓아도 결국 선형 변환 하나와 동일
# - 대표적: relu, sigmoid, tanh, softmax 등
from tensorflow.keras.layers import Activation

# relu: ReLU 활성화 함수 (Rectified Linear Unit)
# - f(x) = max(0, x)
# - x > 0이면 그대로 통과, x ≤ 0이면 0
# - 장점: 계산이 빠르고, 기울기 소실 문제를 sigmoid/tanh보다 잘 완화
# - 현재 딥러닝에서 가장 널리 쓰이는 활성화 함수
from tensorflow.keras.activations import relu

# ============================================================
# 시각화 라이브러리
# ============================================================

# matplotlib: 파이썬의 대표적인 그래프 시각화 라이브러리
import matplotlib.pyplot as plt

print(f"TensorFlow 버전: {tf.__version__}")
print(f"GPU 사용 가능: {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## Cell 2. 랜덤 시드 설정

실험의 **재현성(reproducibility)**을 위해 난수 시드를 고정합니다.

신경망의 가중치 초기화, 데이터 셔플 등에 난수가 사용되므로,
시드를 고정하지 않으면 실행할 때마다 결과가 달라집니다.

In [ ]:
# numpy의 난수 생성기 시드 고정
# - train_test_split의 데이터 분할, 배열 셔플 등에 영향
np.random.seed(0)

# tensorflow의 난수 생성기 시드 고정
# - 가중치 초기화(Xavier, He 등), 드롭아웃 마스크 생성 등에 영향
tf.random.set_seed(0)

---
## Cell 3. 데이터 불러오기 & 탐색

### Wine 데이터셋
- 이탈리아 한 지역에서 재배한 3가지 품종의 와인 178개 샘플
- **피처(X)**: 알코올 도수, 산도, 색상 등 13가지 화학 성분
- **타깃(y)**: 와인 품종 3종 (0, 1, 2)

In [ ]:
# 와인 데이터셋 로드
# - sklearn에 내장된 UCI Wine 데이터셋
# - 반환값: Bunch 객체 (딕셔너리와 유사)
#   - .data: 피처 데이터 (numpy 배열)
#   - .target: 타깃 레이블 (numpy 배열)
#   - .feature_names: 피처 이름 리스트
#   - .target_names: 클래스 이름 리스트
raw_wine = datasets.load_wine()

# 피처 데이터 (X): 입력 변수
# - 신경망이 학습하는 "문제" 부분
# - 각 행 = 와인 1개 샘플, 각 열 = 화학 성분 1가지
X = raw_wine.data

# 타깃 데이터 (y): 정답 레이블
# - 신경망이 맞추려는 "정답" 부분
# - 0, 1, 2 중 하나의 정수값 (와인 품종)
y = raw_wine.target

# 피처 데이터 차원 확인
# - (178, 13) = 178개 샘플, 13개 피처
# - 신경망 입력 차원(input_dim)을 이 값(13)으로 설정해야 함
print(f"피처 데이터 shape: {X.shape}")
print(f"  → {X.shape[0]}개 샘플, {X.shape[1]}개 피처")

# 타깃 데이터 종류 확인
# - set()으로 고유한 값들만 추출
# - {0, 1, 2} = 3가지 클래스
# - 출력층 뉴런 수를 이 값(3)으로 설정해야 함
print(f"\n타깃 클래스: {set(y)}")
print(f"  → {len(set(y))}개 클래스")

# 피처 이름 확인 (어떤 화학 성분들인지)
print(f"\n피처 목록: {raw_wine.feature_names}")

# 처음 3개 샘플 미리보기
print(f"\n처음 3개 샘플 (X):")
print(X[:3])
print(f"\n처음 10개 정답 (y): {y[:10]}")

---
## Cell 4. 데이터 전처리

### 4.1 원-핫 인코딩 (One-Hot Encoding)

분류 문제에서 타깃 레이블을 신경망에 넣으려면 **원-핫 벡터**로 변환해야 합니다.

| 정수 레이블 | 원-핫 벡터 | 의미 |
|:-----------:|:---------:|:----:|
| 0 | [1, 0, 0] | 클래스 0 |
| 1 | [0, 1, 0] | 클래스 1 |
| 2 | [0, 0, 1] | 클래스 2 |

**왜 필요한가?**
- 정수 레이블(0, 1, 2)을 그대로 쓰면 "2가 0보다 크다"는 **순서 관계**가 생김
- 실제로 와인 품종에는 순서가 없으므로, 원-핫 벡터로 **동등한 관계**로 표현
- `categorical_crossentropy` 손실 함수는 원-핫 형태를 요구

### 4.2 데이터 분할 (Train/Test Split)

- **학습 데이터(train)**: 모델이 학습에 사용 (가중치 업데이트)
- **테스트 데이터(test)**: 학습에 사용하지 않고, 모델 성능 평가에만 사용
- 분할하는 이유: **과적합(overfitting)** 감지
  - 학습 데이터에서는 잘 맞추지만 새로운 데이터에서 못 맞추면 과적합

In [ ]:
# 원-핫 인코딩 변환
# - to_categorical(y): 정수 배열 → 원-핫 행렬
# - 입력: [0, 1, 2, 0, ...] (1차원)
# - 출력: [[1,0,0], [0,1,0], [0,0,1], [1,0,0], ...] (2차원)
y_hot = to_categorical(y)

print(f"변환 전 y (처음 5개): {y[:5]}")
print(f"변환 후 y_hot (처음 5개):")
print(y_hot[:5])
print(f"y_hot shape: {y_hot.shape}  → (샘플 수, 클래스 수)")

In [ ]:
# 트레이닝/테스트 데이터 분할
# - test_size: 기본값 0.25 (25%가 테스트, 75%가 학습)
# - random_state=0: 분할 시 셔플 시드 고정 (재현성)
# - 반환 순서: X_train, X_test, y_train, y_test
X_tn, X_te, y_tn, y_te = train_test_split(
    X,        # 피처 데이터 (입력)
    y_hot,    # 타깃 데이터 (원-핫 인코딩된 정답)
    random_state=0  # 셔플 시드 고정
)

print(f"학습 데이터: X_tn {X_tn.shape}, y_tn {y_tn.shape}")
print(f"테스트 데이터: X_te {X_te.shape}, y_te {y_te.shape}")

---
## Cell 5. 신경망 모델 구성

### 모델 구조 설계

```
입력층 (13개 피처)
    ↓
Input(shape=(13,))  ← 입력 형태 정의 (파라미터 없음)
    ↓
Dense(20)           ← 완전연결층: 13차원 → 20차원 (가중치 학습)
    ↓
BatchNormalization  ← 정규화: 학습 안정화
    ↓
Activation('relu')  ← 활성화: 비선형성 부여 (음수 → 0)
    ↓
Dense(3)            ← 완전연결층: 20차원 → 3차원 (클래스 수)
    ↓
Activation('softmax') ← 활성화: 확률 분포로 변환 (합 = 1)
    ↓
출력 [0.05, 0.90, 0.05]  ← 각 클래스에 속할 확률
```

In [ ]:
# ── 하이퍼파라미터 설정 ──────────────────────────────────

# n_feat: 입력 피처 수
# - X_tn.shape = (샘플 수, 피처 수) 이므로 shape[1] = 13
# - Input(shape=(n_feat,)) 레이어에 전달하여 모델 입력 형태 정의
n_feat = X_tn.shape[1]

# n_class: 분류할 클래스 수
# - set(y)로 고유 클래스 개수 계산 = 3
# - 마지막 Dense 레이어의 뉴런 수로 사용
n_class = len(set(y))

# epo (epochs): 전체 학습 데이터를 몇 번 반복 학습할 것인지
# - 1 epoch = 학습 데이터 전체를 한 번 통과
# - 너무 적으면: 학습 부족 (underfitting)
# - 너무 많으면: 과적합 (overfitting)
epo = 30

print(f"입력 피처 수 (n_feat): {n_feat}")
print(f"출력 클래스 수 (n_class): {n_class}")
print(f"에포크 수 (epo): {epo}")

In [ ]:
# ── 신경망 모델 생성 ─────────────────────────────────────

# Sequential(): 레이어를 순서대로 쌓는 모델
# - 가장 간단한 모델 구조 (입력 → 레이어1 → 레이어2 → ... → 출력)
# - add()로 레이어를 하나씩 추가
model = Sequential()

# ── 입력층 (Input Layer) ─────────────────────────────────

# Input(shape=(n_feat,)): 입력 데이터의 형태를 명시적으로 정의
# - shape=(n_feat,) = (13,): 각 샘플이 13개 피처를 가진 1차원 벡터
# - Keras 최신 버전 권장 방식 (Dense(input_dim=...) 사용 시 경고 발생)
# - 이 레이어 자체는 학습 파라미터가 없고, 입력 형태만 알려주는 역할
model.add(Input(shape=(n_feat,)))

# ── 은닉층 (Hidden Layer) ────────────────────────────────

# Dense(20): 첫 번째 완전연결층
# - 20: 이 레이어의 뉴런(노드) 수 = 출력 차원
#   → 20은 설계자가 정하는 하이퍼파라미터 (너무 적으면 표현력 부족, 너무 많으면 과적합)
# - 입력 차원은 바로 위 Input() 레이어에서 자동 추론 (13)
# - 내부 연산: output = W·x + b
#   - W: (13, 20) 크기의 가중치 행렬 = 13×20 = 260개 파라미터
#   - b: (20,) 크기의 편향 벡터 = 20개 파라미터
#   - 이 단계에서는 아직 활성화 함수를 적용하지 않음 (선형 변환만)
model.add(Dense(20))

# BatchNormalization(): 배치 정규화
# - Dense의 출력값을 정규화 (평균=0, 분산=1로 맞춤)
# - 왜 필요한가?
#   1. Internal Covariate Shift 방지: 레이어를 거칠수록 입력 분포가 바뀌는 문제 완화
#   2. 더 높은 learning rate 사용 가능 → 학습 속도 향상
#   3. 약간의 규제(regularization) 효과 → 과적합 완화
# - 학습 시: 미니배치의 평균/분산 사용
# - 추론 시: 학습 중 계산한 이동 평균/분산 사용
# - 학습 가능한 파라미터: γ(스케일), β(시프트) 각각 20개 = 40개
model.add(BatchNormalization())

# Activation('relu'): ReLU 활성화 함수
# - f(x) = max(0, x)
# - 양수는 그대로, 음수는 0으로 만듦
# - 왜 Dense와 분리했나?
#   → Dense(activation='relu')로 합칠 수도 있지만,
#     BatchNormalization을 Dense와 Activation 사이에 넣기 위해 분리
#   → 순서: Dense → BatchNorm → Activation (권장 패턴)
# - 왜 relu를 쓰나?
#   → sigmoid/tanh는 기울기 소실(vanishing gradient) 문제가 있음
#   → relu는 양수 영역에서 기울기가 1이므로 이 문제를 완화
model.add(Activation('relu'))

# ── 출력층 (Output Layer) ────────────────────────────────

# Dense(n_class): 마지막 완전연결층
# - n_class = 3: 분류할 클래스 수만큼 뉴런 생성
# - 각 뉴런의 출력값 = 해당 클래스에 대한 "점수(logit)"
# - 내부 연산: output = W·x + b
#   - W: (20, 3) 크기의 가중치 행렬 = 60개 파라미터
#   - b: (3,) 크기의 편향 벡터 = 3개 파라미터
model.add(Dense(n_class))

# Activation('softmax'): 소프트맥스 활성화 함수
# - 출력층 전용 활성화 함수 (다중 클래스 분류)
# - 수식: softmax(z_i) = exp(z_i) / Σ exp(z_j)
# - 효과:
#   1. 모든 출력값을 0~1 사이로 변환
#   2. 출력값의 합 = 1 (확률 분포)
# - 예: [2.0, 1.0, 0.1] → [0.659, 0.242, 0.099]
#   → "클래스 0일 확률 65.9%, 클래스 1일 확률 24.2%, 클래스 2일 확률 9.9%"
# - 가장 높은 확률의 클래스를 최종 예측으로 선택
model.add(Activation('softmax'))

In [ ]:
# ── 모형 구조 확인 ───────────────────────────────────────
# model.summary(): 모델의 전체 구조를 표로 출력
# - Layer: 레이어 이름과 종류
# - Output Shape: 각 레이어의 출력 크기 (None = 배치 크기, 동적)
# - Param #: 학습 가능한 파라미터 수
# - Total params: 전체 파라미터 수 (이 모델에서 학습할 숫자들의 총 개수)
model.summary()

---
## Cell 6. 모델 컴파일

모델 구조를 만든 후, **어떻게 학습할 것인지** 설정합니다.

컴파일에서 결정하는 3가지:
1. **손실 함수(loss)**: 예측이 정답과 얼마나 다른지 측정하는 함수
2. **옵티마이저(optimizer)**: 손실을 줄이기 위해 가중치를 어떻게 업데이트할지
3. **평가 지표(metrics)**: 학습 과정에서 모니터링할 지표

In [ ]:
# ── 신경망 컴파일 ────────────────────────────────────────
model.compile(
    # loss='categorical_crossentropy': 범주형 교차 엔트로피 손실 함수
    # - 다중 클래스 분류에서 사용 (원-핫 인코딩된 타깃)
    # - 수식: L = -Σ y_true_i · log(y_pred_i)
    #   - y_true = [0, 1, 0] (정답: 클래스 1)
    #   - y_pred = [0.1, 0.8, 0.1] (모델 예측 확률)
    #   - L = -(0·log0.1 + 1·log0.8 + 0·log0.1) = -log(0.8) ≈ 0.22
    # - 예측이 정답에 가까울수록 손실이 0에 가까움
    # - 예측이 정답과 멀수록 손실이 커짐 (무한대로 발산 가능)
    # - 참고: 정수 레이블(원-핫 아님)이면 'sparse_categorical_crossentropy' 사용
    loss='categorical_crossentropy',

    # optimizer='adam': Adam 옵티마이저
    # - 경사하강법(Gradient Descent)의 발전된 버전
    # - 기본 경사하강법: W = W - lr · ∂L/∂W (lr = learning rate)
    # - Adam의 장점:
    #   1. 학습률을 파라미터별로 적응적으로 조절
    #   2. 모멘텀(momentum): 기울기의 이동 평균 → 진동 감소
    #   3. RMSProp: 기울기 제곱의 이동 평균 → 학습률 자동 조절
    # - 기본 학습률: 0.001
    # - 실무에서 가장 널리 쓰이는 옵티마이저 ("잘 모르겠으면 Adam")
    optimizer='adam',

    # metrics=['accuracy']: 학습 중 정확도를 추적
    # - 정확도 = 정답을 맞춘 샘플 수 / 전체 샘플 수
    # - 이 값은 학습에는 영향 없음 (손실 함수만 학습에 사용)
    # - 사람이 모니터링하기 위한 지표
    # - 여러 지표를 동시에 추적 가능: ['accuracy', 'precision', 'recall']
    metrics=['accuracy']
)

print("모델 컴파일 완료!")
print(f"  손실 함수: categorical_crossentropy")
print(f"  옵티마이저: Adam (lr=0.001)")
print(f"  평가 지표: accuracy")

---
## Cell 7. 모델 학습 (Training)

### 학습 과정 (1 epoch)

```
1. 미니배치(5개 샘플)를 모델에 입력
2. 순전파(Forward): 입력 → Dense → BatchNorm → ReLU → Dense → Softmax → 예측값
3. 손실 계산: categorical_crossentropy(정답, 예측값)
4. 역전파(Backward): 손실에 대한 각 가중치의 기울기(gradient) 계산
5. 가중치 업데이트: Adam 옵티마이저가 기울기를 이용해 W, b 수정
6. 다음 미니배치로 1~5 반복 → 전체 데이터 1회 통과 = 1 epoch
```

In [ ]:
# ── 신경망 학습 ──────────────────────────────────────────
# model.fit(): 모델 학습 실행
hist = model.fit(
    X_tn,          # 학습 입력 데이터 (피처)
    y_tn,          # 학습 정답 데이터 (원-핫 인코딩된 타깃)
    epochs=epo,    # 전체 데이터를 30번 반복 학습
    batch_size=5   # 미니배치 크기 = 5
    # batch_size 설명:
    # - 한 번에 5개 샘플을 모델에 넣고 가중치를 업데이트
    # - 학습 데이터 133개 ÷ 배치 5 = 약 27번 가중치 업데이트/epoch
    # - 작을수록: 업데이트 빈번 → 학습 불안정하지만 일반화 좋을 수 있음
    # - 클수록: 업데이트 적음 → 학습 안정적이지만 로컬 미니멈에 빠질 수 있음
    # - 보통 16, 32, 64, 128 등 2의 거듭제곱 사용 (GPU 메모리 효율)
)
# 반환값 hist: History 객체
# - hist.history['loss']: 에포크별 손실값 리스트
# - hist.history['accuracy']: 에포크별 정확도 리스트
# - 이 값들을 이용해 학습 곡선 그래프를 그림

---
## Cell 8. 모델 평가 (Evaluation)

학습이 끝난 모델의 성능을 평가합니다.

- **학습 데이터 정확도**: 모델이 이미 본 데이터에서의 성능
- **테스트 데이터 정확도**: 모델이 한 번도 보지 못한 데이터에서의 성능
- 학습 정확도 >> 테스트 정확도 → **과적합(overfitting)** 의심

In [ ]:
# ── 트레이닝 데이터 평가 ─────────────────────────────────
# model.evaluate(): 주어진 데이터에 대한 손실값과 지표를 계산
# - 반환값: [loss, accuracy] (compile에서 metrics=['accuracy']로 설정했으므로)
# - [0]: loss값, [1]: accuracy값
train_loss, train_acc = model.evaluate(X_tn, y_tn, verbose=0)
print(f"학습 데이터 정확도: {train_acc:.4f} ({train_acc:.1%})")
print(f"학습 데이터 손실:   {train_loss:.4f}")

# ── 테스트 데이터 평가 ────────────────────────────────────
# - 학습에 사용하지 않은 데이터로 평가
# - 이 값이 진짜 모델의 성능 (일반화 능력)
test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
print(f"\n테스트 데이터 정확도: {test_acc:.4f} ({test_acc:.1%})")
print(f"테스트 데이터 손실:   {test_loss:.4f}")

# 과적합 여부 확인
gap = train_acc - test_acc
print(f"\n학습-테스트 정확도 차이: {gap:.4f}")
if gap > 0.1:
    print("  → 과적합 의심! (학습 데이터에만 잘 맞추고 새 데이터에 약함)")
else:
    print("  → 양호 (과적합 없음)")

---
## Cell 9. 학습 과정 시각화

에포크별 **정확도**와 **손실** 변화를 그래프로 확인합니다.

- 정확도: 에포크가 진행될수록 올라가야 정상 (학습이 잘 되고 있다는 의미)
- 손실: 에포크가 진행될수록 내려가야 정상 (예측이 정답에 가까워지고 있다는 의미)

In [ ]:
# ── 학습 기록 추출 ───────────────────────────────────────

# np.arange(1, epo+1): x축 데이터 (1, 2, 3, ..., 30)
# - arange(시작, 끝): 시작부터 끝-1까지 정수 배열 생성
epoch = np.arange(1, epo + 1)

# hist.history['accuracy']: 에포크별 학습 정확도 리스트
# - fit() 실행 중 매 에포크마다 기록된 값
accuracy = hist.history['accuracy']

# hist.history['loss']: 에포크별 학습 손실 리스트
loss = hist.history['loss']

In [ ]:
# ── 정확도 학습 그래프 ───────────────────────────────────

# plt.plot(x, y, label): 선 그래프 그리기
# - epoch: x축 (에포크 번호)
# - accuracy: y축 (정확도)
# - label: 범례에 표시할 이름
plt.plot(epoch, accuracy, label='accuracy')

# plt.xlabel(): x축 레이블
plt.xlabel('epoch')

# plt.ylabel(): y축 레이블
plt.ylabel('accuracy')

# plt.legend(): 범례 표시 (label에 지정한 이름)
plt.legend()

# plt.title(): 그래프 제목
plt.title('Training Accuracy')

# plt.show(): 그래프 출력
plt.show()

In [ ]:
# ── 손실 학습 그래프 ─────────────────────────────────────

# 'r': 빨간색(red)으로 선 그리기
# - 색상 코드: 'r'=빨강, 'b'=파랑, 'g'=초록, 'k'=검정
# - 손실은 빨간색으로 그려 정확도 그래프와 시각적으로 구분
plt.plot(epoch, loss, 'r', label='loss')

plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.title('Training Loss')
plt.show()

---
## 핵심 개념 정리

### 이 실습에서 사용한 구성 요소

| 구성 요소 | 사용한 것 | 역할 |
|----------|----------|------|
| 모델 구조 | `Sequential` | 레이어를 순서대로 쌓는 기본 구조 |
| 은닉층 | `Dense(20)` | 13차원 입력 → 20차원으로 변환 (학습) |
| 정규화 | `BatchNormalization` | 레이어 출력을 정규화하여 학습 안정화 |
| 은닉층 활성화 | `relu` | 비선형성 부여, 기울기 소실 완화 |
| 출력층 | `Dense(3)` | 20차원 → 3차원(클래스 수)으로 변환 |
| 출력층 활성화 | `softmax` | 출력을 확률 분포로 변환 (합=1) |
| 손실 함수 | `categorical_crossentropy` | 정답과 예측의 차이 측정 |
| 옵티마이저 | `adam` | 적응적 학습률로 가중치 업데이트 |
| 평가 지표 | `accuracy` | 정답률 모니터링 |

### 학습 흐름 요약

```
데이터 준비 → 모델 구성(Sequential) → 컴파일(loss, optimizer) → 학습(fit) → 평가(evaluate)
```